In [2]:
# ============================================
# VAR(1) Fit + Eigenvalue Check
# ============================================

import pandas as pd
import numpy as np
from statsmodels.tsa.api import VAR

# =============================
# Config
# =============================
DATA_PATH = "./merged_var_input.csv"
LAG = 1

OUT_PARAMS = "./var_params_var1.csv"
OUT_ROOTS = "./var_roots_var1.csv"
OUT_RESID_COV = "./var_resid_cov_var1.csv"

# =============================
# 1) Load data
# =============================
df = pd.read_csv(DATA_PATH)

if "Date" in df.columns:
    df = df.drop(columns=["Date"])

# diff 변수만 사용
cols = [c for c in df.columns if c.startswith("dlog_") or c.startswith("d_")]
data = df[cols].dropna()

print("사용 변수:", cols)
print("표본 수:", len(data))
print("lag:", LAG)
print("-" * 50)

# =============================
# 2) VAR 적합
# =============================
model = VAR(data)
results = model.fit(LAG)

print(results.summary())

# =============================
# 3) 계수 저장
# =============================
coef_df = pd.DataFrame(
    results.coefs[0],
    index=cols,
    columns=cols
)
coef_df.to_csv(OUT_PARAMS)

# =============================
# 4) Eigenvalue 계산
# =============================
roots = results.roots
roots_df = pd.DataFrame({
    "root": roots,
    "abs_root": np.abs(roots)
})

roots_df.to_csv(OUT_ROOTS, index=False)

print("\nEigenvalues:")
print(roots_df)

# =============================
# 5) Stability 판정
# =============================
is_stable = np.all(np.abs(roots) < 1)
print("\nStable 여부:", is_stable)

# =============================
# 6) 잔차 공분산 저장
# =============================
resid_cov = pd.DataFrame(
    results.sigma_u,
    index=cols,
    columns=cols
)
resid_cov.to_csv(OUT_RESID_COV)

print("\n저장 완료:")
print(OUT_PARAMS)
print(OUT_ROOTS)
print(OUT_RESID_COV)

사용 변수: ['dlog_SOLVPN', 'dlog_COPPER', 'dlog_DXY', 'd_UST10Y', 'dlog_VIX']
표본 수: 1325
lag: 1
--------------------------------------------------
  Summary of Regression Results   
Model:                         VAR
Method:                        OLS
Date:           Tue, 03, Mar, 2026
Time:                     14:18:37
--------------------------------------------------------------------
No. of Equations:         5.00000    BIC:                   -39.1280
Nobs:                     1324.00    HQIC:                  -39.2015
Log likelihood:           16617.2    FPE:                9.03375e-18
AIC:                     -39.2456    Det(Omega_mle):     8.83181e-18
--------------------------------------------------------------------
Results for equation dlog_SOLVPN
                    coefficient       std. error           t-stat            prob
---------------------------------------------------------------------------------
const                  0.000383         0.000339            1.128      